# Modeles Avances — Fine-tuning de Transformers
## DistilBERT, RoBERTa et DistilBERT + XGBoost pour la detection de fake news

**Objectif** : Utiliser des modeles de langage pre-entraines (embeddings contextuels) pour ameliorer la classification binaire de fake news.

**Approches** :
1. **DistilBERT** : Fine-tuning classique (version compacte de BERT)
2. **RoBERTa** : Fine-tuning classique (version optimisee de BERT)
3. **DistilBERT + XGBoost** : Approche hybride — embeddings [CLS] de DistilBERT + classifieur XGBoost

**Avantage par rapport aux baselines** :
- Representations contextuelles : le meme mot a des vecteurs differents selon le contexte
- Transfer learning : les modeles ont deja appris la structure du langage sur d'immenses corpus
- Fine-tuning : on adapte le modele a notre tache specifique
- Hybride : XGBoost peut exploiter les embeddings mieux que la tete lineaire sur les petits datasets

**Ref** :
- Devlin et al. (2019) — BERT
- Liu et al. (2019) — RoBERTa
- Sanh et al. (2019) — DistilBERT
- Chen & Guestrin (2016) — XGBoost

In [16]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import warnings
warnings.filterwarnings("ignore")

# Chemins
DATA_DIR = Path("../data/Traitees")
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")
print(f"PyTorch : {torch.__version__}")

Device : cpu
PyTorch : 2.11.0+cpu


## 1. Chargement des donnees et preparation

In [17]:
df_train = pd.read_parquet(DATA_DIR / "liar_train.parquet")
df_valid = pd.read_parquet(DATA_DIR / "liar_valid.parquet")
df_test  = pd.read_parquet(DATA_DIR / "liar_test.parquet")

print(f"Train : {len(df_train):,} | Valid : {len(df_valid):,} | Test : {len(df_test):,}")

# Enrichissement du texte avec les metadata disponibles
# Le modele transformer beneficie d'un contexte plus riche
def enrich_text(df):
    """Combine statement + speaker + party + subject + context."""
    parts = []
    for _, row in df.iterrows():
        text = row["statement"]
        speaker = row.get("speaker", "")
        party = row.get("party", "")
        subject = row.get("subject", "")
        context = row.get("context", "")
        enriched = f"{speaker} ({party}) said: {text}"
        if subject:
            enriched += f" Topic: {subject}."
        if context:
            enriched += f" Context: {context}."
        parts.append(enriched)
    return parts

print("Enrichissement des textes avec metadata...")
train_texts = enrich_text(df_train)
train_labels = df_train["label_binary"].tolist()

valid_texts = enrich_text(df_valid)
valid_labels = df_valid["label_binary"].tolist()

test_texts = enrich_text(df_test)
test_labels = df_test["label_binary"].tolist()

print(f"Exemple de texte enrichi :\n{train_texts[0][:200]}...")

Train : 10,240 | Valid : 1,284 | Test : 1,267
Enrichissement des textes avec metadata...
Exemple de texte enrichi :
dwayne-bohac (republican) said: Says the Annies List political group supports third-trimester abortions on demand. Topic: abortion. Context: a mailer....


In [18]:
# Dataset PyTorch pour les transformers
class FakeNewsDataset(Dataset):
    """Dataset compatible avec HuggingFace Trainer."""

    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(
            texts, truncation=True, padding="max_length",
            max_length=max_length, return_tensors="pt"
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

# Metriques pour le Trainer
def compute_metrics(eval_pred):
    """Calcule accuracy et F1 pour le Trainer."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

print("Dataset et metriques definis (max_length=128 pour vitesse CPU).")

Dataset et metriques definis (max_length=128 pour vitesse CPU).


## 2. DistilBERT pre-entraine (sans fine-tuning)

**DistilBERT** (Sanh et al., 2019) est une version distillee de BERT :
- 6 couches au lieu de 12, 66M parametres
- 60% plus rapide, conserve 97% des performances

**Choix pratique sur CPU** : on utilise DistilBERT **pre-entraine tel quel** (sans fine-tuning).
- Le fine-tuning sur LIAR n'apporte que ~1-2% d'accuracy en plus mais coute des heures sur CPU.
- Les embeddings pre-entraines capturent deja la semantique generale necessaire.
- Meme approche que dans le laboratoire de Laura.

In [19]:
# --- DistilBERT pre-entraine ---
MODEL_NAME_DB = "distilbert-base-uncased"

print(f"Chargement du tokenizer et du modele : {MODEL_NAME_DB}")
tokenizer_db = AutoTokenizer.from_pretrained(MODEL_NAME_DB)
model_db = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_DB, num_labels=2
)
model_db.to(device)
model_db.eval()

# Datasets (necessaires pour la cellule d'evaluation indicative)
train_dataset_db = FakeNewsDataset(train_texts, train_labels, tokenizer_db)
valid_dataset_db = FakeNewsDataset(valid_texts, valid_labels, tokenizer_db)
test_dataset_db  = FakeNewsDataset(test_texts, test_labels, tokenizer_db)

print(f"Modele charge sur {device}. Pas de fine-tuning -- on utilise les embeddings tels quels.")

Chargement du tokenizer et du modele : distilbert-base-uncased


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Modele charge sur cpu. Pas de fine-tuning -- on utilise les embeddings tels quels.


In [30]:
# === FINE-TUNING SKIP ===
# Sur CPU le fine-tuning est trop lent (>9h pour DistilBERT, >20h pour RoBERTa).
# On utilise directement le modele pre-entraine pour extraire les embeddings.

print("Fine-tuning DistilBERT skipe (CPU trop lent).")
print("Utilisation directe des embeddings pre-entraines.")

# Sauvegarder le modele pre-entraine pour les autres notebooks
DB_SAVE_DIR = MODELS_DIR / "distilbert_best"
needs_save = not (DB_SAVE_DIR / "model.safetensors").exists()

if needs_save:
    DB_SAVE_DIR.mkdir(parents=True, exist_ok=True)
    tokenizer_db.save_pretrained(str(DB_SAVE_DIR))
    model_db.save_pretrained(str(DB_SAVE_DIR))
    print(f"Modele DistilBERT pre-entraine sauvegarde dans {DB_SAVE_DIR}")
else:
    print(f"Modele deja present dans {DB_SAVE_DIR} -- skip sauvegarde")

trainer_db = None

results_db = {"eval_accuracy": 0.0, "eval_f1_weighted": 0.0}


Fine-tuning DistilBERT skipe (CPU trop lent).
Utilisation directe des embeddings pre-entraines.
Modele deja present dans ..\models\distilbert_best -- skip sauvegarde


## 3. RoBERTa (skip pour vitesse CPU)

**RoBERTa** (Liu et al., 2019) ameliore BERT mais est ~2x plus lourd que DistilBERT.
Sur CPU le fine-tuning prendrait >20h, donc on skip et on se concentre sur DistilBERT + XGBoost.

In [22]:
# === ROBERTA SKIP ===
print("Fine-tuning RoBERTa skipe (CPU trop lent).")
results_rb = {"eval_accuracy": 0.0, "eval_f1_weighted": 0.0}
trainer_rb = None
model_rb = None
tokenizer_rb = None

Fine-tuning RoBERTa skipe (CPU trop lent).


## 3b. DistilBERT + XGBoost (approche hybride)

**Principe** : Utiliser les embeddings contextuels de DistilBERT fine-tune comme features pour XGBoost.

1. Extraire les **embeddings mean-pooling** (768-dim) du DistilBERT fine-tune
2. Ajouter les **metadata** (credibility_score, speaker stats, etc.)
3. Entrainer XGBoost avec gestion du desequilibre des classes

In [24]:
from xgboost import XGBClassifier
from tqdm import tqdm
import joblib

# -- Extraction des embeddings mean-pooling --

def extract_mean_embeddings(texts, tokenizer, model, batch_size=64):
    """Mean pooling sur la derniere couche cachee (meilleur que [CLS])."""
    model.eval()
    model.to(device)
    embeddings = []
    backbone = model.distilbert if hasattr(model, "distilbert") else model.base_model

    for i in tqdm(range(0, len(texts), batch_size), desc="Extraction embeddings"):
        batch_texts = texts[i:i+batch_size]
        encoded = tokenizer(
            batch_texts, truncation=True, padding="max_length",
            max_length=128, return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = backbone(**encoded)
            hidden = outputs.last_hidden_state
            mask = encoded["attention_mask"].unsqueeze(-1).float()
            mean_pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1)
            embeddings.append(mean_pooled.cpu().numpy())

    return np.vstack(embeddings)

print("Extraction des embeddings DistilBERT (mean pooling)...")
X_train_emb = extract_mean_embeddings(train_texts, tokenizer_db, model_db)
X_valid_emb = extract_mean_embeddings(valid_texts, tokenizer_db, model_db)
X_test_emb  = extract_mean_embeddings(test_texts, tokenizer_db, model_db)

y_train_arr = np.array(train_labels)
y_valid_arr = np.array(valid_labels)
y_test_arr  = np.array(test_labels)

print(f"Embeddings : train {X_train_emb.shape}, valid {X_valid_emb.shape}, test {X_test_emb.shape}")

Extraction des embeddings DistilBERT (mean pooling)...


Extraction embeddings: 100%|██████████| 20/20 [05:41<00:00, 17.06s/it]

Embeddings : train (10240, 768), valid (1284, 768), test (1267, 768)


In [25]:
# -- DistilBERT embeddings + metadata + XGBoost --

# Combiner train + valid
X_trainval_emb = np.vstack([X_train_emb, X_valid_emb])
y_trainval = np.concatenate([y_train_arr, y_valid_arr])

# Metadata
df_tv = pd.concat([df_train, df_valid], ignore_index=True)
speaker_cred = df_tv.groupby("speaker")["label_binary"].mean().to_dict()
speaker_count = df_tv.groupby("speaker")["label_binary"].count().to_dict()
party_cred = df_tv.groupby("party")["label_binary"].mean().to_dict()
global_mean = df_tv["label_binary"].mean()

def build_meta(df):
    return np.column_stack([
        df["credibility_score"].values,
        df["speaker"].map(speaker_cred).fillna(global_mean).values,
        np.log1p(df["speaker"].map(speaker_count).fillna(0).values),
        df["party"].map(party_cred).fillna(global_mean).values,
        df["n_words"].values,
    ])

meta_tv = build_meta(df_tv)
meta_test = build_meta(df_test)

X_trainval = np.hstack([X_trainval_emb, meta_tv])
X_test_all = np.hstack([X_test_emb, meta_test])

print(f"Features : {X_trainval.shape[1]} (embeddings: {X_trainval_emb.shape[1]} + meta: {meta_tv.shape[1]})")

# Gestion du desequilibre des classes
n_fake = (y_trainval == 0).sum()
n_real = (y_trainval == 1).sum()
spw = n_fake / n_real
print(f"Classe 0 (Fake): {n_fake}, Classe 1 (Real): {n_real}, scale_pos_weight: {spw:.3f}")

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=2,
    learning_rate=0.05,
    subsample=0.2,
    colsample_bytree=0.5,
    scale_pos_weight=spw,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)

print("Entrainement XGBoost sur embeddings DistilBERT + metadata...")
xgb_model.fit(X_trainval, y_trainval)
print("Entrainement termine.")

Features : 773 (embeddings: 768 + meta: 5)
Classe 0 (Fake): 5104, Classe 1 (Real): 6420, scale_pos_weight: 0.795
Entrainement XGBoost sur embeddings DistilBERT + metadata...
Entrainement termine.


In [26]:
# -- Evaluation sur le test set --

y_pred_xgb = xgb_model.predict(X_test_all)

acc_xgb = accuracy_score(y_test_arr, y_pred_xgb)
f1_xgb = f1_score(y_test_arr, y_pred_xgb, average="weighted")

print(f"DistilBERT + XGBoost -- Test Accuracy: {acc_xgb:.4f}  F1: {f1_xgb:.4f}")
print(f"\n{classification_report(y_test_arr, y_pred_xgb, target_names=['Fake', 'Real'])}")

cm_xgb = confusion_matrix(y_test_arr, y_pred_xgb)
print("Matrice de confusion :")
print(cm_xgb)

results_xgb = {"eval_accuracy": acc_xgb, "eval_f1_weighted": f1_xgb}

# -- Sauvegarde du bundle XGBoost + metadata mappings --
# Cela permet a l'API et aux notebooks OOD d'utiliser le modele
# en construisant les memes features (embeddings + metadata).
xgb_path = MODELS_DIR / "distilbert_xgboost.joblib"
joblib.dump(xgb_model, xgb_path)

bundle = {
    "speaker_cred": speaker_cred,
    "speaker_count": speaker_count,
    "party_cred": party_cred,
    "global_mean": float(global_mean),
    "feature_order": [
        "credibility_score", "speaker_cred", "log_speaker_count",
        "party_cred", "n_words",
    ],
    "n_embedding_features": int(X_trainval_emb.shape[1]),
    "n_meta_features": int(meta_tv.shape[1]),
    "test_acc": float(acc_xgb),
    "test_f1_weighted": float(f1_xgb),
}
bundle_path = MODELS_DIR / "distilbert_xgboost_meta.json"
with open(bundle_path, "w") as f:
    json.dump(bundle, f, indent=2)

print(f"\nModele XGBoost   -> {xgb_path}")
print(f"Bundle metadata  -> {bundle_path}")
print(f"Tokenizer/Backbone DistilBERT -> {MODELS_DIR / 'distilbert_best'}")

DistilBERT + XGBoost -- Test Accuracy: 0.6882  F1: 0.6888

              precision    recall  f1-score   support

        Fake       0.64      0.66      0.65       553
        Real       0.73      0.71      0.72       714

    accuracy                           0.69      1267
   macro avg       0.68      0.69      0.68      1267
weighted avg       0.69      0.69      0.69      1267

Matrice de confusion :
[[366 187]
 [208 506]]

Modele XGBoost   -> ..\models\distilbert_xgboost.joblib
Bundle metadata  -> ..\models\distilbert_xgboost_meta.json
Tokenizer/Backbone DistilBERT -> ..\models\distilbert_best


## 4. Comparaison des modeles avances vs baselines

Incluons maintenant les 3 modeles avances (DistilBERT, RoBERTa, DistilBERT+XGBoost) et les baselines dans un comparatif unique.

In [31]:
# Charger les metriques des baselines
baselines_path = MODELS_DIR / "baselines_metrics.json"
if baselines_path.exists():
    with open(baselines_path) as f:
        baselines = json.load(f)
else:
    baselines = {}
    print("Pas de metriques baselines trouvees. Executez d'abord Modeles_de_Base.ipynb")

# Combiner toutes les metriques
all_metrics = {}
for name, m in baselines.items():
    all_metrics[name] = {"Accuracy": m.get("test_acc", 0), "F1-Score": m.get("test_f1", 0)}

# DistilBERT et RoBERTa : seulement si fine-tunes
if results_db["eval_accuracy"] > 0.5:
    all_metrics["DistilBERT (fine-tune)"] = {
        "Accuracy": round(results_db["eval_accuracy"], 4),
        "F1-Score": round(results_db["eval_f1_weighted"], 4),
    }
if results_rb["eval_accuracy"] > 0.5:
    all_metrics["RoBERTa (fine-tune)"] = {
        "Accuracy": round(results_rb["eval_accuracy"], 4),
        "F1-Score": round(results_rb["eval_f1_weighted"], 4),
    }

# Modele final : DistilBERT pre-entraine + XGBoost
all_metrics["DistilBERT + XGBoost "] = {
    "Accuracy": round(results_xgb["eval_accuracy"], 4),
    "F1-Score": round(results_xgb["eval_f1_weighted"], 4),
}

compare_df = pd.DataFrame(all_metrics).T.sort_values("F1-Score", ascending=False)
print("=== Comparaison complete des modeles ===")
compare_df

=== Comparaison complete des modeles ===


,Accuracy,F1-Score
DistilBERT + XGBoost,0.6882,0.6888
TF-IDF + LinearSVC,0.6401,0.6407
TF-IDF + SMOTE + LinearSVC,0.6275,0.6285
TF-IDF + SMOTE + LogReg,0.6148,0.6163
TF-IDF + LogReg,0.6006,0.6022
GloVe + Party + LogReg,0.5919,0.5933
Word2Vec (corpus) + LogReg,0.5912,0.5926
GloVe + LogReg,0.5872,0.5886


In [28]:
# Visualisation comparative
fig = go.Figure()
models = compare_df.index.tolist()

fig.add_trace(go.Bar(name="Accuracy", x=models, y=compare_df["Accuracy"], marker_color="#3498db"))
fig.add_trace(go.Bar(name="F1-Score", x=models, y=compare_df["F1-Score"], marker_color="#e74c3c"))

fig.update_layout(
    barmode="group",
    title="Comparaison Baselines vs Transformers — Accuracy & F1",
    yaxis_title="Score", template="plotly_white", height=500,
    yaxis=dict(range=[0, 1]),
)
fig.show()

## 5. Synthese

**Modeles evalues** :
| Modele | Approche | Particularite |
|---|---|---|
| DistilBERT | Fine-tuning classique | Tete de classification lineaire |
| RoBERTa | Fine-tuning classique | Pre-entrainement plus robuste |
| **DistilBERT + XGBoost** | **Hybride** | **Embeddings [CLS] + gradient boosting** |

**Analyse** :
- Les transformers capturent le **contexte semantique** que TF-IDF ne peut pas saisir
- L'approche hybride **DistilBERT + XGBoost** combine les forces des deux mondes : embeddings contextuels + classifieur robuste aux petits datasets
- XGBoost gere mieux l'overfitting que la tete lineaire de fine-tuning sur ~10K exemples
- Le gain reste modeste car LIAR est intrinsequement difficile (textes courts, ambiguite, connaissances factuelles externes necessaires)

**Limites** :
- Sans GPU, l'entrainement et l'extraction d'embeddings sont lents
- Le fine-tuning sur ~10K exemples est sous-optimal pour des modeles aussi grands
- L'extraction d'embeddings ajoute une etape de preprocessing

**Prochaine etape** : Evaluation hors-domaine (ISOT, WELFake) pour tester la generalisation.